# Run CRISE on Synthetic Probes (Goal 6)

Runs `run_crise()` on every valid synthetic probe in `metadata.csv`.  
Saliency maps are cached to `results/crise_maps/` (same directory as real-probe maps).  
Saves a saliency index CSV mapping `output_path → saliency_path` for use in forensics analysis.

**Resumable**: re-running this notebook skips already-completed probes.  
**Run order**: after all three `synth_probe_*.ipynb` notebooks have completed.

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

META_PATH       = "data/synthetic_probes/metadata.csv"
GALLERY_EMB     = "cache/G.npy"
GALLERY_IDS     = "cache/gallery_ids.npy"

CRISE_OUT_DIR   = "results/crise_maps"
SAL_INDEX_PATH  = "results/crise_maps/synth_saliency_index.csv"

SYNTH_SEED      = 42
N               = 1000
S               = 8
P1              = 0.5
TAU             = 0.1

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis

from crise import run_crise, TAU

os.makedirs(CRISE_OUT_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load gallery
# ---------------------------------------------------------------------------

G           = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}
print(f"Gallery: {G.shape[0]} identities")

In [ ]:
# ---------------------------------------------------------------------------
# CRISE loop
# ---------------------------------------------------------------------------

SAVE_EVERY = 25   # write sal_index to disk every N completed probes

sal_index = []
n_done  = 0
n_fail  = 0

for idx, row in tqdm(valid.iterrows(), total=len(valid), desc="CRISE on synthetic probes"):
    run_seed = SYNTH_SEED * 10_000 + idx

    try:
        result = run_crise(
            true_id     = row["identity"],
            probe_path  = row["output_path"],
            G           = G,
            id_to_index = id_to_index,
            rec         = rec,
            app         = app,
            run_seed    = run_seed,
            out_dir     = CRISE_OUT_DIR,
            tau         = TAU,
            N           = N,
            s           = S,
            p1          = P1,
        )
        sal_index.append(dict(
            output_path       = row["output_path"],
            identity          = row["identity"],
            generation_method = row["generation_method"],
            blend_alpha       = row["blend_alpha"],
            saliency_path     = result["saliency_path"],
            w_clean           = result["w_clean"],
            failures          = result["failures"],
        ))
        n_done += 1

        # Periodic checkpoint — survives kernel crash
        if n_done % SAVE_EVERY == 0:
            pd.DataFrame(sal_index).to_csv(SAL_INDEX_PATH, index=False)
            print(f"  [checkpoint] {n_done}/{len(valid)} saved → {SAL_INDEX_PATH}")

    except Exception as e:
        print(f"[FAIL] {row['identity']} | {row['output_path']}: {e}")
        n_fail += 1

print(f"\nDone: {n_done} saliency maps computed, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# CRISE loop
# ---------------------------------------------------------------------------

sal_index = []   # list of {output_path, saliency_path, w_clean, failures}
n_done  = 0
n_fail  = 0

for idx, row in tqdm(valid.iterrows(), total=len(valid), desc="CRISE on synthetic probes"):
    run_seed = SYNTH_SEED * 10_000 + idx

    try:
        result = run_crise(
            true_id    = row["identity"],
            probe_path = row["output_path"],
            G          = G,
            id_to_index= id_to_index,
            rec        = rec,
            app        = app,
            run_seed   = run_seed,
            out_dir    = CRISE_OUT_DIR,
            tau        = TAU,
            N          = N,
            s          = S,
            p1         = P1,
        )
        sal_index.append(dict(
            output_path   = row["output_path"],
            identity      = row["identity"],
            generation_method = row["generation_method"],
            blend_alpha   = row["blend_alpha"],
            saliency_path = result["saliency_path"],
            w_clean       = result["w_clean"],
            failures      = result["failures"],
        ))
        n_done += 1
    except Exception as e:
        print(f"[FAIL] {row['identity']} | {row['output_path']}: {e}")
        n_fail += 1

print(f"\nDone: {n_done} saliency maps computed, {n_fail} failed")

In [ ]:
# ---------------------------------------------------------------------------
# Save saliency index
# ---------------------------------------------------------------------------

sal_df = pd.DataFrame(sal_index)
sal_df.to_csv(SAL_INDEX_PATH, index=False)
print(f"Saliency index: {len(sal_df)} entries → {SAL_INDEX_PATH}")
print(sal_df.groupby("generation_method").size().to_string())